In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.data import besInferenceDatapoints
from neuro_bes.preprocessing.profile_transform import InterpolateToCommonGrid

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.interpolate import interp1d
import pandas as pd

In [ ]:
path="/home/molnarbalazs/data/BES_ML_modelling/W7X_op23/test_newcuration"
file_list=os.listdir(path)
file_list=[i for i in file_list if "_w" in i]

In [ ]:
len(file_list)

In [ ]:
batch_slowmod=[]
batch_fastmod=[]
batch=[]
for db_file in file_list:
    bes_data=besInferenceDatapoints(path=os.path.join(path,db_file))
    if bes_data.ID=="we_20250513.018":
        bes_data_20250513018_we=besInferenceDatapoints(path=os.path.join(path,db_file))
    if bes_data.ID=="ws_20250513.018":
        bes_data_20250513018_ws=besInferenceDatapoints(path=os.path.join(path,db_file))
    if bes_data.ID=="we_20250514.066":
        bes_data_20250514066_we=besInferenceDatapoints(path=os.path.join(path,db_file))
    if bes_data.emissions.shape[0]>1:
        if "fast modulation" in bes_data.verbose:
            batch_fastmod.append(bes_data)
        if "slow modulation" in bes_data.verbose:
            batch_slowmod.append(bes_data)
        batch.append(bes_data)
    else:
        print(f"skipping {bes_data.ID} with empty data")

In [ ]:
batch_fastmod[0].emissions.shape

In [ ]:
number_of_fast_profiles=0
for b in batch_fastmod:
    number_of_fast_profiles += b.emissions.shape[0]
print(f"number of fast modulation profiles: {number_of_fast_profiles}")
number_of_slow_profiles=0
for b in batch_slowmod:
    number_of_slow_profiles += b.emissions.shape[0]
print(f"number of slow modulation profiles: {number_of_slow_profiles}")

In [ ]:
plt.figure(figsize=(20,5))
plt.subplot(1,2,1)
random_batches=np.random.randint(0,len(batch_slowmod),10)
for idx in random_batches:
    b = batch_slowmod[int(idx)]
    row = np.random.randint(0, b.emissions.shape[0],10)
    plt.plot(b.grid, b.emissions[row].T)
plt.subplot(1,2,2)
random_batches=np.random.randint(0,len(batch_fastmod),10)
for idx in random_batches:    
    b = batch_fastmod[int(idx)]
    row = np.random.randint(0, b.emissions.shape[0],10)
    plt.plot(b.grid, b.emissions[row].T)
plt.show()

In [ ]:

plt.figure(figsize=(20,5))
colors = plt.cm.viridis(np.linspace(0, 1, bes_data_20250513018_we.densities.shape[0]))
plt.subplot(1,4,1)
for i in range(bes_data_20250513018_we.emissions.shape[0]):
    plt.plot(bes_data_20250513018_we.grid, bes_data_20250513018_we.emissions[i].T, color=colors[i])
plt.subplot(1,4,2)
for i in range(bes_data_20250513018_we.densities.shape[0]):
    plt.plot(bes_data_20250513018_we.grid, bes_data_20250513018_we.densities[i].T, color=colors[i])
plt.subplot(1,4,3)
for i in range(bes_data_20250513018_ws.emissions.shape[0]):
    plt.plot(bes_data_20250513018_ws.grid, bes_data_20250513018_ws.emissions[i].T, color=colors[i])
plt.subplot(1,4,4)
for i in range(bes_data_20250513018_ws.densities.shape[0]):
    plt.plot(bes_data_20250513018_ws.grid, bes_data_20250513018_ws.densities[i].T, color=colors[i])

In [ ]:
plt.figure(figsize=(20,5))
colors = plt.cm.viridis(np.linspace(0, 1, bes_data_20250514066_we.densities.shape[0]))
plt.subplot(1,4,1)
for i in range(bes_data_20250514066_we.emissions.shape[0]):
    plt.plot(bes_data_20250514066_we.grid, bes_data_20250514066_we.emissions[i].T, color=colors[i])
plt.subplot(1,4,2)
for i in range(bes_data_20250514066_we.densities.shape[0]):
    plt.plot(bes_data_20250514066_we.grid, bes_data_20250514066_we.densities[i].T, color=colors[i])

In [ ]:
bes_data_20250514066_we.emissions[:,0]

In [ ]:
filepath='/home/molnarbalazs/data/BES_ML_modelling/logbook_summary'
#process rows of txt file
with open(filepath, 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()]
logbook=pd.DataFrame(columns=["field","gas","ratio","ratio_error"])
for i, logbook_entry in enumerate(lines):
    shot=logbook_entry.split(" - ")[0]
    field=logbook_entry.split(" - ")[1].split(" --- ")[0].split(": ")[1]
    gas=logbook_entry.split(" - ")[1].split(" --- ")[1].split(": ")[1]
    ratio=logbook_entry.split(" - ")[1].split(" --- ")[2].split(": ")[1].split(" +- ")[0]
    ratio_error=logbook_entry.split(" - ")[1].split(" --- ")[2].split(": ")[1].split(" +- ")[1]
    logbook.loc[i,"shot"]=shot
    logbook.loc[i,"field"]=field
    logbook.loc[i,"gas"]=gas
    if str(ratio).strip().lower() in ('unknown', ''):
        logbook.loc[i, "ratio"] = np.nan
    else:
        logbook.loc[i, "ratio"] = np.float32(ratio)

    if str(ratio_error).strip().lower() in ('unknown', ''):
        ratio_error = np.nan
    logbook.loc[i,"ratio_error"]=np.float32(ratio_error)

In [ ]:
logbook.head()

In [ ]:
logbook=logbook.dropna()

In [ ]:
logbook.field.unique()

In [ ]:
shot_H=list(logbook[logbook.ratio>0.95]["shot"].values)
shot_notH=list(logbook[logbook.ratio<0.5]["shot"].values)
shot_EIM=list(logbook[logbook.field.str.contains("EIM")]["shot"].values)
shot_KJM=list(logbook[logbook.field.str.contains("KJM")]["shot"].values)

In [ ]:
len(batch)

In [ ]:
batch[1].ID[3:]

In [ ]:
profiles_d = batch[1].densities
profiles_e = batch[1].emissions
n_profiles = profiles_d.shape[0]
colors = plt.cm.viridis(np.linspace(0, 1, n_profiles))

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
for i, profile in enumerate(profiles_d):
    plt.plot(batch[0].grid, profile, color=colors[i])
plt.title("densities")

plt.subplot(1, 2, 2)
for i, profile in enumerate(profiles_e):
    plt.plot(batch[0].grid, profile, color=colors[i])
plt.title("emissions")

In [ ]:
interpolator=InterpolateToCommonGrid()
batch_interp=interpolator.fit_transform(batch)
emission_H=np.concatenate([b.emissions for b in batch_interp if b.ID[3:] in shot_H])
density_H=np.concatenate([b.densities for b in batch_interp if b.ID[3:] in shot_H])
emission_notH=np.concatenate([b.emissions for b in batch_interp if b.ID[3:] in shot_notH])
density_notH=np.concatenate([b.densities for b in batch_interp if b.ID[3:] in shot_notH])
emission_EIM=np.concatenate([b.emissions for b in batch_interp if b.ID[3:] in shot_EIM])
density_EIM=np.concatenate([b.densities for b in batch_interp if b.ID[3:] in shot_EIM])
emission_KJM=np.concatenate([b.emissions for b in batch_interp if b.ID[3:] in shot_KJM])
density_KJM=np.concatenate([b.densities for b in batch_interp if b.ID[3:] in shot_KJM])
r_coord=interpolator.common_grid_

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(10,8))
for i in (np.random.rand(100)*emission_H.shape[0]).astype(int):
    ax[0,0].plot(r_coord,emission_H[i])
for i in (np.random.rand(100)*emission_notH.shape[0]).astype(int):
    ax[0,1].plot(r_coord,emission_notH[i])
for i in (np.random.rand(100)*density_H.shape[0]).astype(int):
    ax[1,0].plot(r_coord,density_H[i])
for i in (np.random.rand(100)*density_notH.shape[0]).astype(int):
    ax[1,1].plot(r_coord,density_notH[i])
ax[0,0].set_title("H-rich shots")
ax[0,1].set_title("H-poor shots")
ax[1,0].set_title("H-rich density")
ax[1,1].set_title("H-poor density")
ax[0,0].set_xlabel("r [m]")
ax[0,0].set_ylabel("emission [arb. units]")
ax[0,1].set_xlabel("r [m]")
ax[0,1].set_ylabel("emission [arb. units]")
ax[1,0].set_xlabel("r [m]")
ax[1,0].set_ylabel("density [arb. units]")
ax[1,1].set_xlabel("r [m]")
ax[1,1].set_ylabel("density [arb. units]")
plt.tight_layout()
plt.show()

In [ ]:
EIM_profiles=np.random.rand(100)*emission_EIM.shape[0]
KJM_profiles=np.random.rand(100)*emission_KJM.shape[0]
fig,ax=plt.subplots(1,2,figsize=(10,5),sharex=True,sharey=True)
for i in EIM_profiles.astype(int):
    ax[0].plot(r_coord,emission_EIM[i])
for i in KJM_profiles.astype(int):
    ax[1].plot(r_coord,emission_KJM[i])
ax[0].set_title("EIM shots")
ax[1].set_title("KJM shots")
ax[0].set_ylabel("emission [arb. units]")
ax[1].set_xlabel("r [m]")
plt.tight_layout()
plt.show()

fig,ax=plt.subplots(1,2,figsize=(10,5),sharex=True,sharey=True)
for i in EIM_profiles.astype(int):
    ax[0].plot(r_coord,density_EIM[i])
for i in KJM_profiles.astype(int):
    ax[1].plot(r_coord,density_KJM[i])
ax[0].set_title("EIM density")
ax[1].set_title("KJM density")
ax[1].set_xlabel("r [m]")
ax[0].set_ylabel("density [arb. units]")
plt.tight_layout()
plt.show()